In [ ]:
import sys
!{sys.executable} -m pip install matplotlib seaborn plotly --upgrade

In [11]:
#!/usr/bin/env python3
"""
multi_factor_screener.py
=========================
Standalone script version of the Multi-Factor Stock Screener.

This mirrors the logic built out cell-by-cell in
`Multi_Factor_Stock_Screener.ipynb`, refactored into an importable module
with a CLI entry point so it can run outside Colab — in a cron job, a CI
pipeline, or your own IDE.

Usage
-----
    python multi_factor_screener.py                     # demo mode, 60 names
    python multi_factor_screener.py --full-universe      # full S&P 500
    python multi_factor_screener.py --top-n 30 --output-dir results/

Outputs
-------
    <output_dir>/factor_screen_<timestamp>.csv   full ranked universe
    <output_dir>/top_<N>_<timestamp>.csv         shortlist
    <output_dir>/charts/*.png                    static charts
    <output_dir>/charts/score_vs_risk.html       interactive Plotly chart

See the project README.md for full methodology, architecture, and
disclosed limitations (especially around the factor-validation backtest).
"""

from __future__ import annotations

import argparse
import datetime as dt
import logging
import os
import time
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("multi_factor_screener")


# ======================================================================
# CONFIGURATION
# ======================================================================
@dataclass
class ScreenerConfig:
    """Every tunable knob for the screener lives here — no magic numbers
    buried in the pipeline itself."""

    # --- Universe ---
    demo_mode: bool = True
    demo_sample_size: int = 60

    # --- Screening filters ---
    min_market_cap_usd: float = 2_000_000_000
    min_avg_dollar_volume: float = 5_000_000

    # --- Factor lookback windows ---
    momentum_lookback_days: int = 252
    momentum_skip_days: int = 21
    volatility_lookback_days: int = 252
    price_history_period: str = "2y"

    # --- Factor weights (must sum to 1.0) ---
    factor_weights: Dict[str, float] = field(default_factory=lambda: {
        "value": 0.25,
        "quality": 0.25,
        "growth": 0.20,
        "momentum": 0.20,
        "low_volatility": 0.10,
    })

    # --- Scoring behaviour ---
    sector_neutral: bool = True
    winsorize_limits: Tuple[float, float] = (0.01, 0.01)

    # --- Output ---
    top_n: int = 20
    output_dir: str = "screener_output"

    # --- Networking ---
    request_pause_sec: float = 0.15

    def validate(self) -> None:
        total_w = round(sum(self.factor_weights.values()), 4)
        if total_w != 1.0:
            raise ValueError(f"Factor weights must sum to 1.0, got {total_w}")
        logger.info("Config validated. Factor weights: %s", self.factor_weights)


FUNDAMENTAL_FIELDS = [
    "marketCap", "trailingPE", "priceToBook", "enterpriseToEbitda",
    "returnOnEquity", "returnOnAssets", "grossMargins", "debtToEquity",
    "revenueGrowth", "earningsGrowth", "earningsQuarterlyGrowth",
    "beta", "averageDailyVolume10Day", "shortName",
]

FALLBACK_UNIVERSE = pd.DataFrame({
    "Symbol": ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "BRK-B", "JPM", "V", "UNH",
               "XOM", "JNJ", "PG", "MA", "HD", "MRK", "ABBV", "COST", "PEP", "KO",
               "AVGO", "WMT", "BAC", "CVX", "LLY", "ADBE", "CRM", "TMO", "MCD", "CSCO"],
    "GICS Sector": ["Information Technology", "Information Technology", "Communication Services",
                     "Consumer Discretionary", "Information Technology", "Communication Services",
                     "Financials", "Financials", "Financials", "Health Care", "Energy", "Health Care",
                     "Consumer Staples", "Financials", "Consumer Discretionary", "Health Care",
                     "Health Care", "Consumer Staples", "Consumer Staples", "Consumer Staples",
                     "Information Technology", "Consumer Staples", "Financials", "Energy",
                     "Health Care", "Information Technology", "Information Technology", "Health Care",
                     "Consumer Discretionary", "Information Technology"],
})


# ======================================================================
# STEP 1 — UNIVERSE CONSTRUCTION
# ======================================================================
def get_sp500_universe(demo_mode: bool = True, sample_size: int = 60,
                        random_state: int = 42) -> pd.DataFrame:
    """Fetch S&P 500 tickers + GICS sector from Wikipedia, with a static fallback."""
    try:
        tables = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")
        sp500 = tables[0][["Symbol", "GICS Sector"]].copy()
        logger.info("Fetched %d tickers from Wikipedia S%26P 500 table.", len(sp500))
    except Exception as e:
        logger.warning("Wikipedia scrape failed (%s). Using fallback universe.", e)
        sp500 = FALLBACK_UNIVERSE.copy()

    sp500.columns = ["ticker", "sector"]
    sp500["ticker"] = sp500["ticker"].str.replace(".", "-", regex=False)
    sp500 = sp500.drop_duplicates(subset="ticker").reset_index(drop=True)

    if demo_mode and len(sp500) > sample_size:
        sp500 = sp500.sample(n=sample_size, random_state=random_state).reset_index(drop=True)
        logger.info("Demo mode — sampled %d tickers.", sample_size)

    return sp500


# ======================================================================
# STEP 2 — DATA COLLECTION
# ======================================================================
def fetch_fundamentals(tickers: List[str], pause_sec: float = 0.15) -> pd.DataFrame:
    """Batch-fetch fundamentals; a single bad ticker never aborts the batch."""
    import yfinance as yf  # imported lazily so this module can be linted/tested without it installed

    records, failed = [], []
    for i, ticker in enumerate(tickers, start=1):
        for attempt in range(3):
            try:
                info = yf.Ticker(ticker).info
                row = {"ticker": ticker}
                row.update({f: info.get(f, np.nan) for f in FUNDAMENTAL_FIELDS})
                records.append(row)
                break
            except Exception as e:
                if attempt == 2:
                    logger.warning("Fundamentals fetch failed for %s: %s", ticker, e)
                    failed.append(ticker)
                else:
                    time.sleep(2 ** attempt)
        time.sleep(pause_sec)
        if i % 10 == 0 or i == len(tickers):
            logger.info("Fundamentals progress: %d/%d", i, len(tickers))

    if failed:
        logger.warning("Could not retrieve fundamentals for %d tickers: %s", len(failed), failed)
    return pd.DataFrame(records)


def fetch_price_history(tickers: List[str], period: str = "2y") -> pd.DataFrame:
    """One batched download for all tickers (fast, API-friendly)."""
    import yfinance as yf

    try:
        raw = yf.download(tickers=tickers, period=period, interval="1d", auto_adjust=True,
                           group_by="ticker", threads=True, progress=False)
    except Exception as e:
        logger.error("Batch price download failed entirely: %s", e)
        return pd.DataFrame()

    close_frames = {}
    for ticker in tickers:
        try:
            series = raw[ticker]["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw["Close"]
            if series.dropna().empty:
                raise ValueError("empty series")
            close_frames[ticker] = series
        except Exception:
            logger.warning("No usable price history for %s — dropped.", ticker)

    prices = pd.DataFrame(close_frames).sort_index()
    logger.info("Price history: %d tickers x %d trading days.", prices.shape[1], prices.shape[0])
    return prices


# ======================================================================
# STEP 3 — CLEANING
# ======================================================================
def build_master_frame(universe: pd.DataFrame, fundamentals: pd.DataFrame,
                        prices: pd.DataFrame, config: ScreenerConfig) -> pd.DataFrame:
    df = universe.merge(fundamentals, on="ticker", how="inner")

    valid_price_tickers = set(prices.columns)
    df = df[df["ticker"].isin(valid_price_tickers)].reset_index(drop=True)

    for col in [c for c in FUNDAMENTAL_FIELDS if c != "shortName"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    last_close = prices[df["ticker"]].iloc[-1]
    df["dollar_volume"] = (last_close.values * df["averageDailyVolume10Day"].values)

    df = df[(
        df["marketCap"] >= config.min_market_cap_usd) &
            (df["dollar_volume"] >= config.min_avg_dollar_volume)].reset_index(drop=True)

    core_required = ["trailingPE", "priceToBook", "returnOnEquity", "debtToEquity"]
    df = df.dropna(subset=core_required, how="all").reset_index(drop=True)

    logger.info("Master frame ready: %d tickers x %d columns.", *df.shape)
    return df


# ======================================================================
# STEP 4 — FEATURE ENGINEERING
# ======================================================================
def compute_momentum(prices: pd.DataFrame, tickers: List[str],
                      lookback: int, skip: int) -> pd.Series:
    px = prices[tickers].dropna(how="all")
    if len(px) <= lookback + skip:
        lookback = max(len(px) - skip - 1, 1)
    end_px = px.iloc[-skip - 1]
    start_idx = -(lookback + skip + 1)
    start_px = px.iloc[start_idx] if len(px) > abs(start_idx) else px.iloc[0]
    return ((end_px / start_px) - 1.0).rename("momentum_raw")


def compute_realized_volatility(prices: pd.DataFrame, tickers: List[str],
                                  lookback: int) -> pd.Series:
    px = prices[tickers].dropna(how="all")
    rets = np.log(px / px.shift(1)).tail(lookback)
    return (rets.std() * np.sqrt(252)).rename("volatility_raw")


def engineer_features(master: pd.DataFrame, prices: pd.DataFrame,
                       config: ScreenerConfig) -> pd.DataFrame:
    df = master.copy()
    tickers = df["ticker"].tolist()

    momentum = compute_momentum(prices, tickers,
                                config.momentum_lookback_days, config.momentum_skip_days)
    volatility = compute_realized_volatility(prices, tickers, config.volatility_lookback_days)

    df = df.merge(momentum, left_on="ticker", right_index=True, how="left")
    df = df.merge(volatility, left_on="ticker", right_index=True, how="left")

    df["value_pe"] = df["trailingPE"]
    df["value_pb"] = df["priceToBook"]
    df["value_ev_ebitda"] = df["enterpriseToEbitda"]
    df["quality_roe"] = df["returnOnEquity"]
    df["quality_roa"] = df["returnOnAssets"]
    df["quality_margin"] = df["grossMargins"]
    df["quality_leverage"] = df["debtToEquity"]
    df["growth_revenue"] = df["revenueGrowth"]
    df["growth_earnings"] = df["earningsGrowth"].fillna(df["earningsQuarterlyGrowth"])
    df["momentum"] = df["momentum_raw"]
    df["volatility"] = df["volatility_raw"]
    return df


# ======================================================================
# STEP 5 — SCORING ENGINE
# ======================================================================
def winsorize(series: pd.Series, limits: Tuple[float, float]) -> pd.Series:
    lower, upper = series.quantile(limits[0]), series.quantile(1 - limits[1])
    return series.clip(lower, upper)


def sector_zscore(df: pd.DataFrame, column: str, sector_col: str = "sector") -> pd.Series:
    def _z(group: pd.Series) -> pd.Series:
        if group.notna().sum() < 3 or group.std(ddof=0) == 0:
            return pd.Series(np.nan, index=group.index)
        return (group - group.mean()) / group.std(ddof=0)

    scored = df.groupby(sector_col)[column].transform(_z)
    mu, sigma = df[column].mean(), df[column].std(ddof=0)
    fallback = (df[column] - mu) / sigma if sigma else 0
    return scored.fillna(fallback)


class FactorScoringEngine:
    """Raw metrics -> sector-neutral z-scores -> weighted composite score."""

    RAW_COLUMNS = {
        "value": ["value_pe", "value_pb", "value_ev_ebitda"],
        "quality": ["quality_roe", "quality_roa", "quality_margin", "quality_leverage"],
        "growth": ["growth_revenue", "growth_earnings"],
        "momentum": ["momentum"],
        "low_volatility": ["volatility"],
    }
    INVERT_METRICS = {"value_pe", "value_pb", "value_ev_ebitda", "quality_leverage", "volatility"}

    def __init__(self, config: ScreenerConfig):
        self.config = config

    def score(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        sector_col = "sector" if self.config.sector_neutral else None

        for factor, raw_cols in self.RAW_COLUMNS.items():
            z_cols = []
            for col in raw_cols:
                clean = winsorize(out[col], self.config.winsorize_limits)
                if sector_col:
                    z = sector_zscore(out.assign(**{col: clean}), col, sector_col)
                else:
                    mu, sigma = clean.mean(), clean.std(ddof=0)
                    z = (clean - mu) / sigma if sigma else pd.Series(0, index=clean.index)
                if col in self.INVERT_METRICS:
                    z = -z
                out[f"z_{col}"] = z
                z_cols.append(f"z_{col}")
            out[f"factor_{factor}"] = out[z_cols].mean(axis=1, skipna=True)

        weights = self.config.factor_weights
        out["composite_score"] = sum(
            out[f"factor_{factor}"].fillna(0) * w for factor, w in weights.items()
        )
        out["rank"] = out["composite_score"].rank(ascending=False, method="min").astype(int)
        return out.sort_values("rank").reset_index(drop=True)


# ======================================================================
# STEP 6 — CHARTS (saved to disk — headless-safe for CLI/CI use)
# ======================================================================
def generate_charts(scored_df: pd.DataFrame, config: ScreenerConfig, charts_dir: str) -> None:
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns

    os.makedirs(charts_dir, exist_ok=True)
    n = config.top_n
    factor_cols = ["factor_value", "factor_quality", "factor_growth",
                   "factor_momentum", "factor_low_volatility"]
    factor_labels = ["Value", "Quality", "Growth", "Momentum", "Low Vol"]

    # --- Top-N composite score bar chart ---
    plot_df = scored_df.head(n).sort_values("composite_score")
    fig, ax = plt.subplots(figsize=(9, max(5, n * 0.32)))
    ax.barh(plot_df["ticker"], plot_df["composite_score"],
            color=sns.color_palette("mako", n_colors=len(plot_df)))
    ax.axvline(0, color="#333333", linewidth=0.8)
    ax.set_title(f"Top {n} Stocks — Composite Score", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(charts_dir, "top_composite_scores.png"))
    plt.show()

    # --- Factor exposure heatmap ---
    heat_df = scored_df.head(n).set_index("ticker")[factor_cols]
    heat_df.columns = factor_labels
    fig, ax = plt.subplots(figsize=(7, max(5, n * 0.35)))
    sns.heatmap(heat_df, annot=True, fmt=".2f", cmap="RdYlGn", center=0, ax=ax)
    ax.set_title(f"Factor Exposure Heatmap — Top {n}", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(charts_dir, "factor_heatmap.png"))
    plt.show()

    # --- Sector breakdown ---
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sector_counts = scored_df.head(n)["sector"].value_counts()
    axes[0].pie(sector_counts.values, labels=sector_counts.index, autopct="%1.0f%%",
                colors=sns.color_palette("mako", n_colors=len(sector_counts)))
    axes[0].set_title(f"Sector Mix — Top {n}", fontweight="bold")
    sector_avg = scored_df.groupby("sector")["composite_score"].mean().sort_values()
    axes[1].barh(sector_avg.index, sector_avg.values,
                 color=np.where(sector_avg.values >= 0, "#2c7a4b", "#b23b3b"))
    axes[1].axvline(0, color="#333333", linewidth=0.8)
    axes[1].set_title("Avg. Composite Score by Sector", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(charts_dir, "sector_breakdown.png"))
    plt.show()

    # --- Correlation matrix ---
    corr = scored_df[factor_cols].corr()
    corr.index = corr.columns = factor_labels
    fig, ax = plt.subplots(figsize=(6, 5))
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                vmin=-1, vmax=1, ax=ax)
    ax.set_title("Factor Correlation Matrix", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(charts_dir, "factor_correlation.png"))
    plt.show()

    # --- Interactive Plotly dashboard (saved as standalone HTML) ---
    try:
        import plotly.express as px
        fig = px.scatter(scored_df, x="volatility", y="composite_score", color="sector",
                          size="marketCap", hover_name="ticker", template="plotly_white",
                          title="Composite Score vs. Realized Volatility")
        fig.write_html(os.path.join(charts_dir, "score_vs_risk.html"))
        fig.show()
    except ImportError:
        logger.warning("plotly not installed — skipping interactive dashboard export.")


# ======================================================================
# STEP 7 — VALIDATION BACKTEST (see README for disclosed limitations)
# ======================================================================
def run_quintile_backtest(prices: pd.DataFrame, scored: pd.DataFrame,
                           forward_days: int = 126) -> pd.DataFrame:
    tickers = scored["ticker"].tolist()
    px = prices[tickers].dropna(how="all")
    if len(px) <= forward_days + 10:
        logger.warning("Not enough history for a %d-day forward test; skipping.", forward_days)
        return pd.DataFrame()

    forward_return = (px.iloc[-1] / px.iloc[-forward_days - 1]) - 1.0
    bt = scored[["ticker", "composite_score"]].copy()
    bt["forward_return"] = bt["ticker"].map(forward_return)
    bt = bt.dropna(subset=["forward_return"])
    bt["quintile"] = pd.qcut(bt["composite_score"], 5, labels=[1, 2, 3, 4, 5], duplicates="drop")
    return bt


def compute_performance_metrics(backtest_df: pd.DataFrame) -> dict:
    if backtest_df.empty:
        return {}
    from scipy import stats

    ic, ic_p = stats.spearmanr(backtest_df["composite_score"], backtest_df["forward_return"])
    top_q = backtest_df[backtest_df["quintile"] == backtest_df["quintile"].max()]["forward_return"]
    bot_q = backtest_df[backtest_df["quintile"] == backtest_df["quintile"].min()]["forward_return"]
    hit_rate = (backtest_df.assign(
        pred=lambda d: d["composite_score"] > d["composite_score"].median(),
        actual=lambda d: d["forward_return"] > d["forward_return"].median(),
    ).eval("pred == actual").mean())

    return {
        "information_coefficient": round(float(ic), 4),
        "ic_p_value": round(float(ic_p), 4),
        "quintile_spread": float(top_q.mean() - bot_q.mean()),
        "hit_rate": float(hit_rate),
        "universe_size": len(backtest_df),
    }


# ======================================================================
# PIPELINE ORCHESTRATION
# ======================================================================
def run_pipeline(config: ScreenerConfig) -> pd.DataFrame:
    """Runs the full screener end-to-end and returns the scored, ranked DataFrame."""
    config.validate()
    os.makedirs(config.output_dir, exist_ok=True)
    charts_dir = os.path.join(config.output_dir, "charts")

    universe_df = get_sp500_universe(config.demo_mode, config.demo_sample_size)
    fundamentals_df = fetch_fundamentals(universe_df["ticker"].tolist(), config.request_pause_sec)
    price_history_df = fetch_price_history(universe_df["ticker"].tolist(), config.price_history_period)

    master_df = build_master_frame(universe_df, fundamentals_df, price_history_df, config)
    featured_df = engineer_features(master_df, price_history_df, config)

    engine = FactorScoringEngine(config)
    scored_df = engine.score(featured_df)

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M")
    full_path = os.path.join(config.output_dir, f"factor_screen_{timestamp}.csv")
    top_path = os.path.join(config.output_dir, f"top_{config.top_n}_{timestamp}.csv")
    scored_df.to_csv(full_path, index=False)
    scored_df.head(config.top_n).to_csv(top_path, index=False)
    logger.info("Exported: %s | %s", full_path, top_path)

    backtest_df = run_quintile_backtest(price_history_df, scored_df)
    metrics = compute_performance_metrics(backtest_df)
    if metrics:
        logger.info("Validation metrics: %s", metrics)

    return scored_df, charts_dir


# ======================================================================
# CLI ENTRY POINT
# ======================================================================
def main() -> None:
    parser = argparse.ArgumentParser(description="Multi-Factor Stock Screener")
    parser.add_argument("--full-universe", action="store_true",
                         help="Screen the full S%26P 500 instead of the fast demo sample.")
    parser.add_argument("--top-n", type=int, default=20, help="Number of stocks in the shortlist.")
    parser.add_argument("--output-dir", type=str, default="screener_output",
                         help="Directory for CSVs and charts.")
    args = parser.parse_args([])

    config = ScreenerConfig(
        demo_mode=not args.full_universe,
        top_n=args.top_n,
        output_dir=args.output_dir,
    )
    scored_df, charts_dir = run_pipeline(config)
    print(scored_df[["rank", "ticker", "sector", "composite_score"]].head(config.top_n).to_string(index=False))

    try:
        generate_charts(scored_df, config, charts_dir)
    except Exception as e:
        logger.warning("Chart generation skipped due to error: %s", e)


if __name__ == "__main__":
    main()

 rank ticker                 sector  composite_score
    1   NVDA Information Technology         0.741065
    2  GOOGL Communication Services         0.557078
    3     KO       Consumer Staples         0.411123
    4    LLY            Health Care         0.386827
    5  BRK-B             Financials         0.386067
    6    MCD Consumer Discretionary         0.354736
    7    MRK            Health Care         0.345796
    8    JNJ            Health Care         0.291963
    9    PEP       Consumer Staples         0.270338
   10   AMZN Consumer Discretionary         0.100049
   11   META Communication Services         0.073723
   12    BAC             Financials         0.073107
   13    JPM             Financials         0.042300
   14   CSCO Information Technology         0.013152
   15    XOM                 Energy         0.003623
   16   MSFT Information Technology         0.000048
   17    CVX                 Energy        -0.027979
   18   ADBE Information Technology        -0.